<a href="https://colab.research.google.com/github/susara20010420/Workplace-Safety-Insights-from-the-Industrial-Safety-Health-Analytics-Dataset/blob/main/WHSAT_text_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets scikit-learn torch

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

In [25]:
df = pd.read_csv("https://raw.githubusercontent.com/susara20010420/Workplace-Safety-Insights-from-the-Industrial-Safety-Health-Analytics-Dataset/main/cleaned_safety_data_set_A.csv")
# Keep only necessary columns
data_set = df[['description', 'severity', 'critical_risk']].dropna()
data_set.head()

,description,severity,critical_risk
0,While removing the drill rod of the Jumbo 08 f...,1,Pressed
1,During the activation of a sodium sulphide pum...,1,Pressurized Systems
2,In the sub-station MILPO located at level +170...,1,Manual Tools
3,Being 9:45 am. approximately in the Nv. 1880 C...,1,Others
4,Approximately at 11:45 a.m. in circumstances t...,4,Others


In [30]:
data_set['critical_risk_label'] = data_set['critical_risk'].astype('category').cat.codes
risk_labels = dict(enumerate(data_set['critical_risk'].astype('category').cat.categories))
print(risk_labels)

{0: '\nNot applicable', 1: 'Bees', 2: 'Blocking and isolation of energies', 3: 'Burn', 4: 'Chemical substances', 5: 'Confined space', 6: 'Cut', 7: 'Electrical Shock', 8: 'Electrical installation', 9: 'Fall', 10: 'Fall prevention', 11: 'Fall prevention (same level)', 12: 'Individual protection equipment', 13: 'Liquid Metal', 14: 'Machine Protection', 15: 'Manual Tools', 16: 'Others', 17: 'Plates', 18: 'Poll', 19: 'Power lock', 20: 'Pressed', 21: 'Pressurized Systems', 22: 'Pressurized Systems / Chemical Substances', 23: 'Projection', 24: 'Projection of fragments', 25: 'Projection/Burning', 26: 'Projection/Choco', 27: 'Projection/Manual Tools', 28: 'Suspended Loads', 29: 'Traffic', 30: 'Vehicles and Mobile Equipment', 31: 'Venomous Animals', 32: 'remains of choco'}


In [31]:
#to make 0–5
data_set['severity_label'] = data_set['severity'] - 1

In [32]:
#splitting the data set as trainning category (80%) and testing category (20%)
train_df, test_df = train_test_split(data_set, test_size=0.2, stratify=data_set['severity_label'], random_state=42)

In [35]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=128
    )

In [38]:
from torch.utils.data import Dataset

class SafetyDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenize(texts)
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [39]:
train_dataset = SafetyDataset(train_df['description'], train_df['critical_risk_label'])
test_dataset = SafetyDataset(test_df['description'], test_df['critical_risk_label'])

num_labels = data_set['critical_risk_label'].nunique()

In [40]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [41]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,          # keep small for CPU
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_dir='./logs',
    save_strategy="no"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [42]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "f1": f1_score(labels, preds, average='macro')
    }

In [43]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,F1
1,No log,2.695191,0.030075
2,No log,2.637962,0.030075


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=86, training_loss=2.176081102947856, metrics={'train_runtime': 565.9673, 'train_samples_per_second': 1.201, 'train_steps_per_second': 0.152, 'total_flos': 22531907450880.0, 'train_loss': 2.176081102947856, 'epoch': 2.0})